# OmniVoice Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb)

This notebook demonstrates the basic usage of [OmniVoice](https://github.com/k2-fsa/OmniVoice), a massively multilingual zero-shot TTS model supporting 600+ languages.

**Contents:**
1. Installation
2. Option A — Gradio Demo (interactive web UI, no code needed)
3. Option B — Python API
   - 3.1 Load Model
   - 3.2 Voice Cloning
   - 3.3 Voice Design
   - 3.4 Auto Voice

## 1. Installation

Colab already provides a compatible PyTorch + CUDA environment, so we only need to install OmniVoice.

In [4]:
!pip install omnivoice

  Using cached omnivoice-0.2.1-py3-none-any.whl.metadata (16 kB)
  Using cached tensorboardx-2.6.5-py3-none-any.whl.metadata (6.6 kB)
  Using cached webdataset-1.0.2-py3-none-any.whl.metadata (12 kB)
  Using cached braceexpand-0.1.7-py2.py3-none-any.whl.metadata (3.0 kB)
Using cached omnivoice-0.2.1-py3-none-any.whl (168 kB)
Using cached tensorboardx-2.6.5-py3-none-any.whl (87 kB)
Using cached webdataset-1.0.2-py3-none-any.whl (74 kB)
Using cached braceexpand-0.1.7-py2.py3-none-any.whl (5.9 kB)


## 2. Option A — Gradio Demo

Launch an interactive web UI with a public Gradio link. The `--share` flag creates a temporary public URL so you can access the demo from any browser.

> **If you prefer to use the Python API directly, skip to Option B below.**

In [2]:
!omnivoice-demo --share

/bin/bash: line 1: omnivoice-demo: command not found


## 3. Option B — Python API

### 3.1 Load Model

In [5]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

### 3.2 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [24]:
from google.colab import files

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())
print(f"Uploaded: {ref_audio_path}")

Upload a reference audio file (wav/mp3/flac):


Saving common_voice_pt_19275115.mp3 to common_voice_pt_19275115 (2).mp3
Saving common_voice_pt_19276292.mp3 to common_voice_pt_19276292 (2).mp3
Saving common_voice_pt_19276297.mp3 to common_voice_pt_19276297 (2).mp3
Uploaded: ['common_voice_pt_19275115 (2).mp3', 'common_voice_pt_19276292 (2).mp3', 'common_voice_pt_19276297 (2).mp3']


In [26]:
sentences = ["Quero que você aumente meu limite!", "Você pode aumentar meu limite?", "Quero consultar meu saldo."]
transcriptions = ["O esposo dela é engenheiro.", "Alguém pode atender o telefone?", "A alça da taça está quebrada." ]

audio = model.generate(
    text=sentences,
    ref_audio=ref_audio_path,
    ref_text=transcriptions,
)

print(len(audio))
sf.write("clone_out1.wav", audio[0], 24000)
sf.write("clone_out2.wav", audio[1], 24000)
sf.write("clone_out3.wav", audio[2], 24000)
display(Audio(audio[0], rate=24000))
display(Audio(audio[1], rate=24000))
display(Audio(audio[2], rate=24000))

3


### 3.3 Voice Design

Describe the desired voice with speaker attributes — no reference audio needed.

Supported attributes: gender, age, pitch, style (whisper), English accent, Chinese dialect. See [docs/voice-design.md](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design.md) for the full list.

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 Auto Voice

Let the model choose a voice automatically — no reference audio or instruct needed.

In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))